# 05 — Dashboard data prep

Builds Streamlit inputs under **`dashboard_data/`** by **reusing** outputs from notebook **03** (`airline_delay_summary.csv`, `airport_delay_summary.csv`, `seasonal_delay_summary.csv`) and **04** (`airport_clusters.csv`, `cluster_summary.csv`). No flight-level CSV is required.

In [7]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

In [8]:
ROOT = Path(".")
OUT = ROOT / "dashboard_data"
OUT.mkdir(parents=True, exist_ok=True)

CAUSES = [
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
]

## Airline dashboard slice ← `airline_delay_summary.csv` (notebook 03)

In [9]:
src = ROOT / "airline_delay_summary.csv"
if not src.exists():
    raise FileNotFoundError("Run notebook 03 first to create airline_delay_summary.csv")

a = pd.read_csv(src)
out = pd.DataFrame({
    "airline": a["Reporting_Airline"],
    "flights": a["flights"],
})
for c in CAUSES:
    out[f"total_{c}"] = a[f"total_{c}"]
    out[f"share_{c}"] = a[f"pct_{c}"] / 100.0

out.to_csv(OUT / "dashboard_airline_summary.csv", index=False)
out.head()

,airline,flights,total_CarrierDelay,share_CarrierDelay,total_WeatherDelay,share_WeatherDelay,total_NASDelay,share_NASDelay,total_SecurityDelay,share_SecurityDelay,total_LateAircraftDelay,share_LateAircraftDelay,pct_delayed,avg_arr_delay
0,9E,19154,68701.0,0.3165,11388.0,0.0525,59717.0,0.2751,216.0,0.0010,77013.0,0.3548,NaN,NaN
1,AA,230329,452736.0,0.3121,84959.0,0.0586,399173.0,0.2751,2110.0,0.0015,511806.0,0.3528,NaN,NaN
2,AS,48923,80617.0,0.3230,5671.0,0.0227,63704.0,0.2552,1445.0,0.0058,98165.0,0.3933,NaN,NaN
3,B6,37033,163731.0,0.2790,16573.0,0.0282,169815.0,0.2894,1479.0,0.0025,235201.0,0.4008,NaN,NaN
4,CO,89698,70856.0,0.2354,11476.0,0.0381,138405.0,0.4598,1022.0,0.0034,79232.0,0.2632,NaN,NaN


## Airport dashboard slice ← `airport_delay_summary.csv` + `airport_clusters.csv` (03 / 04)

In [10]:
src = ROOT / "airport_delay_summary.csv"
if not src.exists():
    raise FileNotFoundError("Run notebook 03 first to create airport_delay_summary.csv")

p = pd.read_csv(src)
o = pd.DataFrame({
    "airport": p["Origin"].astype(str),
    "role": "Origin",
    "flights": p["flights"],
    "pct_delayed": p["delay_rate"],
    "dominant_cause": p["dominant_cause"],
})
for c in CAUSES:
    o[f"total_{c}"] = p[f"total_{c}"]

ts = o[[f"total_{c}" for c in CAUSES]].sum(axis=1).replace(0, np.nan)
for c in CAUSES:
    o[f"share_{c}"] = (o[f"total_{c}"] / ts).fillna(0)

clus = ROOT / "airport_clusters.csv"
if clus.exists():
    cl = pd.read_csv(clus)[["Origin", "cluster"]].rename(columns={"Origin": "airport"})
    o = o.merge(cl, on="airport", how="left")
else:
    o["cluster"] = np.nan

o.to_csv(OUT / "dashboard_airport_summary.csv", index=False)
o.head()

,airport,role,flights,pct_delayed,dominant_cause,avg_arr_delay,total_CarrierDelay,total_WeatherDelay,total_NASDelay,total_SecurityDelay,total_LateAircraftDelay,share_CarrierDelay,share_WeatherDelay,share_NASDelay,share_SecurityDelay,share_LateAircraftDelay,cluster
0,ABE,Origin,1596,0.402256,NASDelay,NaN,2866.0,204.0,2928.0,0.0,2826.0,0.324796,0.023119,0.331822,0.000000,0.320263,NaN
1,ABI,Origin,462,0.309524,CarrierDelay,NaN,1784.0,339.0,578.0,0.0,1199.0,0.457436,0.086923,0.148205,0.000000,0.307436,NaN
2,ABQ,Origin,10779,0.423138,LateAircraftDelay,NaN,10365.0,1809.0,8193.0,52.0,23859.0,0.234089,0.040856,0.185035,0.001174,0.538845,6.0
3,ABR,Origin,80,0.275000,CarrierDelay,NaN,1839.0,73.0,66.0,9.0,346.0,0.788255,0.031290,0.028290,0.003858,0.148307,NaN
4,ABY,Origin,182,0.489011,CarrierDelay,NaN,1088.0,108.0,789.0,0.0,913.0,0.375431,0.037267,0.272257,0.000000,0.315045,NaN


## Monthly & seasonal dashboard slices ← `seasonal_delay_summary.csv` (notebook 03)

In [11]:
src = ROOT / "seasonal_delay_summary.csv"
if not src.exists():
    raise FileNotFoundError("Run notebook 03 first to create seasonal_delay_summary.csv")

s = pd.read_csv(src)

m = s[s["time_grain"] == "month"].copy()
m["Month"] = m["period"].astype(int)
month_out = pd.DataFrame({"Month": m["Month"]})
for c in CAUSES:
    month_out[f"total_{c}"] = m[f"total_{c}"].values
    month_out[f"avg_{c}"] = m[f"avg_{c}"].values

month_out.to_csv(OUT / "dashboard_monthly_summary.csv", index=False)

ss = s[s["time_grain"] == "season"].copy()
ss["Season"] = ss["period"].astype(str)
sea_out = pd.DataFrame({"Season": ss["Season"]})
for c in CAUSES:
    sea_out[f"total_{c}"] = ss[f"total_{c}"].values
    sea_out[f"avg_{c}"] = ss[f"avg_{c}"].values

order = ["Winter", "Spring", "Summer", "Autumn"]
sea_out["Season"] = pd.Categorical(sea_out["Season"], categories=order, ordered=True)
sea_out = sea_out.sort_values("Season")
sea_out.to_csv(OUT / "dashboard_seasonal_summary.csv", index=False)

month_out.head(), sea_out

(   Month  total_CarrierDelay  avg_CarrierDelay  total_WeatherDelay  \
 0      1            316121.0          1.921241             70754.0   
 1      2            296594.0          1.958350             59163.0   
 2      3            316035.0          1.849012             41577.0   
 3      4            278439.0          1.730746             46922.0   
 4      5            279875.0          1.699746             50593.0   
 
    avg_WeatherDelay  total_NASDelay  avg_NASDelay  total_SecurityDelay  \
 0          0.430011        288040.0      1.750577               1545.0   
 1          0.390641        262894.0      1.735835               1661.0   
 2          0.243253        275069.0      1.609334               1357.0   
 3          0.291662        252849.0      1.571682                977.0   
 4          0.307263        281570.0      1.710040                846.0   
 
    avg_SecurityDelay  total_LateAircraftDelay  avg_LateAircraftDelay  flights  \
 0           0.009390                 

## Cluster files ← `cluster_summary.csv` (notebook 04); copy scatter file

In [12]:
cs = ROOT / "cluster_summary.csv"
if cs.exists():
    df = pd.read_csv(cs)
    df.to_csv(OUT / "dashboard_cluster_summary.csv", index=False)

    cent_cols = [c for c in df.columns if c.startswith("centroid_")]
    long_rows = []
    for _, row in df.iterrows():
        for cc in cent_cols:
            long_rows.append({
                "cluster": int(row["cluster"]),
                "metric": cc.replace("centroid_", ""),
                "value": float(row[cc]),
            })
    pd.DataFrame(long_rows).to_csv(OUT / "dashboard_cluster_centroids.csv", index=False)
else:
    pd.DataFrame(columns=["cluster", "n_airports", "dominant_delay_type", "sample_airports"]).to_csv(
        OUT / "dashboard_cluster_summary.csv", index=False
    )
    pd.DataFrame(columns=["cluster", "metric", "value"]).to_csv(
        OUT / "dashboard_cluster_centroids.csv", index=False
    )

ac = ROOT / "airport_clusters.csv"
if ac.exists():
    shutil.copy2(ac, OUT / "airport_clusters.csv")

sorted(p.name for p in OUT.iterdir())

['airport_clusters.csv',
 'dashboard_airline_summary.csv',
 'dashboard_airport_summary.csv',
 'dashboard_cluster_centroids.csv',
 'dashboard_cluster_summary.csv',
 'dashboard_monthly_summary.csv',
 'dashboard_seasonal_summary.csv']